# 05 — Epoching

Creates time-locked epoch files for each window defined in the YAML config.
Each window produces an independent FIF file.

Baseline correction and peak-to-peak rejection are intentionally skipped here
and applied downstream after ICA and artifact rejection.

**Input:** `<subject>_preprocessed_clean_raw.fif`, `<subject>_derived_events_eve.fif`  
**Output:** `<subject>_<window>-epo.fif` for each window in `epoching_windows`

In [ ]:
%load_ext autoreload
%autoreload 2

from eeg_toolkit import load_config, find_subjects, create_all_windows

# ── Update this path to point to your experiment config ──
cfg = load_config('../configs/your_experiment.yaml')

subjects = find_subjects(cfg)
print(f"Subjects: {len(subjects)}")

In [ ]:
# ── Test with the first subject ──
test_subject = subjects[0]
print(f"\nEpoching {test_subject}\n")

success = create_all_windows(cfg, test_subject, overwrite=True, verbose=True)
print(f"\nResult: {'OK' if success else 'skipped or failed'}")

In [ ]:
# ── Verify each window ──
import mne
from eeg_toolkit import get_epochs_path

for window in cfg.epoching_windows:
    path = get_epochs_path(cfg, test_subject, window.name)
    if path.exists():
        epochs = mne.read_epochs(path, preload=False, verbose='WARNING')
        size_mb = path.stat().st_size / 1e6
        print(f"=== {window.name} ===")
        print(f"  file:    {path.name} ({size_mb:.1f} MB)")
        print(f"  epochs:  {len(epochs)}")
        print(f"  window:  [{epochs.tmin}, {epochs.tmax}] s")
        print(f"  sfreq:   {epochs.info['sfreq']} Hz")
        print(f"  event_id: {epochs.event_id}")
        print()

In [ ]:
# ── Epoch all subjects ──
from eeg_toolkit import epoch_all_subjects

summary = epoch_all_subjects(cfg, overwrite=False, verbose=True)